# PBL — Waste Classification: Model Training Continuation

**What this notebook does (picks up where `PBL_Waste_Classification.ipynb` left off):**

1. Loads your already-trained baseline (`best_model.pth`, the 5-layer CNN)
2. Trains **MobileNetV2** with transfer learning → addresses Research Gap #3 (lightweight, mobile-deployable)
3. Trains **ResNet18** with transfer learning → second comparison point
4. Builds a **comparison table** (Baseline CNN vs MobileNetV2 vs ResNet18)
5. Adds **ROC curve** and **per-class metrics**
6. Adds **Grad-CAM** heatmaps → addresses Research Gap #4 (explainability)
7. Zips all new outputs so you can download them for the report/PPT

**Requirements:** Runtime → Change runtime type → **T4 GPU**

## Step 1 — Imports and device check

In [ ]:
import os, time, copy, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report,
                             roc_curve, auc)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

OUT_DIR = '/content/outputs_continuation'
os.makedirs(OUT_DIR, exist_ok=True)

## Step 2 — Dataset path

Same folder layout as the first notebook. If your dataset is somewhere else, change `DATASET_DIR`.

In [ ]:
DATASET_DIR = '/content/dataset/dataset/DATASET'
train_dir = os.path.join(DATASET_DIR, 'TRAIN')
test_dir  = os.path.join(DATASET_DIR, 'TEST')

assert os.path.isdir(train_dir), f'TRAIN folder not found at {train_dir}'
assert os.path.isdir(test_dir),  f'TEST folder not found at {test_dir}'
print('TRAIN classes:', sorted(os.listdir(train_dir)))
print('TEST  classes:', sorted(os.listdir(test_dir)))

## Step 3 — DataLoaders

Pretrained ImageNet models expect **224×224** RGB with ImageNet normalization. We use two transform sizes:
- `tf_128` → for the baseline CNN (matches how it was trained)
- `tf_224_train` / `tf_224_eval` → for MobileNetV2 and ResNet18

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
BATCH = 32

# Baseline CNN was trained at 128x128 — keep same for fair re-eval
tf_128 = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Pretrained models: 224x224
tf_224_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
tf_224_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_128  = datasets.ImageFolder(test_dir,  transform=tf_128)
train_224 = datasets.ImageFolder(train_dir, transform=tf_224_train)
test_224  = datasets.ImageFolder(test_dir,  transform=tf_224_eval)

CLASSES = train_224.classes  # ['O', 'R']
print('Classes:', CLASSES, '(index 0 = Organic, index 1 = Recyclable)')
print('Train images:', len(train_224), '| Test images:', len(test_224))

train_loader_224 = DataLoader(train_224, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_loader_224  = DataLoader(test_224,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader_128  = DataLoader(test_128,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## Step 4 — Shared helpers (train / evaluate)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    running_loss, total = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        if criterion is not None:
            running_loss += criterion(out, y).item() * x.size(0)
        probs = torch.softmax(out, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())
        total += x.size(0)
    y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    loss = (running_loss / total) if criterion is not None else None
    return {'loss': loss, 'acc': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob}

def fit(model, train_loader, test_loader, epochs, lr, weight_decay=1e-4, tag='model'):
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.5)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state = 0.0, None
    for e in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
        v = evaluate(model, test_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(v['loss']); history['val_acc'].append(v['acc'])
        if v['acc'] > best_acc:
            best_acc = v['acc']
            best_state = copy.deepcopy(model.state_dict())
        print(f"[{tag}] Epoch {e}/{epochs}  train_loss={tl:.4f} train_acc={ta:.4f}  val_loss={v['loss']:.4f} val_acc={v['acc']:.4f}  ({time.time()-t0:.1f}s)")
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_acc

## Step 5 — Re-load your baseline CNN and evaluate it

Upload `best_model.pth` to Colab first (the file panel on the left → upload). This gets us the baseline numbers for the comparison table.

In [ ]:
class WasteClassifierCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128,256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(256,512, 3, padding=1),nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*4*4, 256), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(256, 2),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

BASELINE_PATH = '/content/best_model.pth'
assert os.path.exists(BASELINE_PATH), 'Upload best_model.pth to /content first (Colab file panel).'

baseline = WasteClassifierCNN().to(device)
state = torch.load(BASELINE_PATH, map_location=device)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
baseline.load_state_dict(state)
baseline_res = evaluate(baseline, test_loader_128, nn.CrossEntropyLoss())
print(f"Baseline CNN — test acc={baseline_res['acc']:.4f}  precision={baseline_res['precision']:.4f}  recall={baseline_res['recall']:.4f}  f1={baseline_res['f1']:.4f}")

## Step 6 — MobileNetV2 (transfer learning)

**Why MobileNetV2?** ~3.5 M parameters, ~14 MB, designed for phones → directly answers Research Gap #3.

Two-phase training:
- **Phase 1 (feature extractor):** freeze backbone, train only the classifier head. 5 epochs, lr=1e-3.
- **Phase 2 (fine-tune):** unfreeze the last few blocks, train with a small lr. 3 epochs, lr=1e-4.

In [ ]:
def build_mobilenet(num_classes=2):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    in_feat = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_feat, num_classes)
    return m

mobilenet = build_mobilenet().to(device)

# Phase 1 — freeze backbone
for p in mobilenet.features.parameters():
    p.requires_grad = False
print('Phase 1 — training classifier head only')
h1_mob, _ = fit(mobilenet, train_loader_224, test_loader_224, epochs=5, lr=1e-3, tag='MobileNetV2/head')

# Phase 2 — unfreeze last few blocks and fine-tune
for name, p in mobilenet.named_parameters():
    if 'features.14' in name or 'features.15' in name or 'features.16' in name or 'features.17' in name or 'features.18' in name or 'classifier' in name:
        p.requires_grad = True
print('Phase 2 — fine-tuning last blocks')
h2_mob, _ = fit(mobilenet, train_loader_224, test_loader_224, epochs=3, lr=1e-4, tag='MobileNetV2/ft')

mob_res = evaluate(mobilenet, test_loader_224, nn.CrossEntropyLoss())
print(f"MobileNetV2 — test acc={mob_res['acc']:.4f}  precision={mob_res['precision']:.4f}  recall={mob_res['recall']:.4f}  f1={mob_res['f1']:.4f}")
torch.save(mobilenet.state_dict(), os.path.join(OUT_DIR, 'mobilenetv2_waste.pth'))

## Step 7 — ResNet18 (transfer learning)

Second comparison model — deeper backbone, slightly larger.

In [ ]:
def build_resnet18(num_classes=2):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

resnet = build_resnet18().to(device)

# Phase 1 — freeze everything except fc
for p in resnet.parameters():
    p.requires_grad = False
for p in resnet.fc.parameters():
    p.requires_grad = True
print('Phase 1 — training fc only')
h1_res, _ = fit(resnet, train_loader_224, test_loader_224, epochs=5, lr=1e-3, tag='ResNet18/head')

# Phase 2 — unfreeze layer4 + fc
for name, p in resnet.named_parameters():
    if name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True
print('Phase 2 — fine-tuning layer4')
h2_res, _ = fit(resnet, train_loader_224, test_loader_224, epochs=3, lr=1e-4, tag='ResNet18/ft')

res_res = evaluate(resnet, test_loader_224, nn.CrossEntropyLoss())
print(f"ResNet18 — test acc={res_res['acc']:.4f}  precision={res_res['precision']:.4f}  recall={res_res['recall']:.4f}  f1={res_res['f1']:.4f}")
torch.save(resnet.state_dict(), os.path.join(OUT_DIR, 'resnet18_waste.pth'))

## Step 8 — Comparison table + chart

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

def size_mb(state_path):
    return os.path.getsize(state_path) / (1024*1024)

torch.save(baseline.state_dict(), os.path.join(OUT_DIR, 'baseline_cnn.pth'))

rows = [
    ('Baseline CNN (5-layer)', count_params(baseline),
     size_mb(os.path.join(OUT_DIR, 'baseline_cnn.pth')),
     baseline_res['acc'], baseline_res['precision'], baseline_res['recall'], baseline_res['f1']),
    ('MobileNetV2 (transfer)',  count_params(mobilenet),
     size_mb(os.path.join(OUT_DIR, 'mobilenetv2_waste.pth')),
     mob_res['acc'], mob_res['precision'], mob_res['recall'], mob_res['f1']),
    ('ResNet18 (transfer)',     count_params(resnet),
     size_mb(os.path.join(OUT_DIR, 'resnet18_waste.pth')),
     res_res['acc'], res_res['precision'], res_res['recall'], res_res['f1']),
]

print(f"{'Model':<26}{'Params':>12}{'Size(MB)':>12}{'Acc':>10}{'Prec':>10}{'Rec':>10}{'F1':>10}")
for r in rows:
    print(f"{r[0]:<26}{r[1]:>12,}{r[2]:>12.2f}{r[3]:>10.4f}{r[4]:>10.4f}{r[5]:>10.4f}{r[6]:>10.4f}")

with open(os.path.join(OUT_DIR, 'comparison.json'), 'w') as f:
    json.dump([{'model': r[0], 'params': r[1], 'size_mb': r[2],
                'accuracy': r[3], 'precision': r[4], 'recall': r[5], 'f1': r[6]} for r in rows], f, indent=2)

labels = [r[0] for r in rows]
accs   = [r[3] for r in rows]
f1s    = [r[6] for r in rows]
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, accs, w, label='Accuracy')
ax.bar(x + w/2, f1s,  w, label='F1 (weighted)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylim(0, 1.0); ax.set_ylabel('Score')
ax.set_title('Model comparison — test set')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, f + 0.01, f'{f:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '08_model_comparison.png'), dpi=150)
plt.show()

## Step 9 — ROC curve and per-class metrics (best model)

Picks whichever of the two transfer-learning models scored higher.

In [ ]:
best_name, best_res = ('MobileNetV2', mob_res) if mob_res['acc'] >= res_res['acc'] else ('ResNet18', res_res)
print('Best transfer-learning model:', best_name)

y_true = best_res['y_true']
y_prob_pos = best_res['y_prob'][:, 1]  # probability of class R
fpr, tpr, _ = roc_curve(y_true, y_prob_pos)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, label=f'{best_name} (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Recyclable class'); axes[0].legend(); axes[0].grid(alpha=0.3)

prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(y_true, best_res['y_pred'], labels=[0, 1], zero_division=0)
x = np.arange(2); w = 0.25
axes[1].bar(x - w, prec_c, w, label='Precision')
axes[1].bar(x,     rec_c,  w, label='Recall')
axes[1].bar(x + w, f1_c,   w, label='F1')
axes[1].set_xticks(x); axes[1].set_xticklabels(['Organic (O)', 'Recyclable (R)'])
axes[1].set_ylim(0, 1.05); axes[1].set_title(f'Per-class metrics — {best_name}')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '09_roc_and_per_class.png'), dpi=150)
plt.show()

print('\nClassification report ({}):\n'.format(best_name))
print(classification_report(y_true, best_res['y_pred'], target_names=['Organic', 'Recyclable'], digits=4))

## Step 10 — Grad-CAM (explainability)

Answers Research Gap #4. Grad-CAM highlights which pixels most influenced the model's prediction — the heatmap turns a black-box CNN into something a human can inspect.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, module, inp, out):
        self.activations = out.detach()
    def _bwd(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()
    def __call__(self, x, class_idx=None):
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # [1, C, 1, 1]
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, torch.softmax(out, dim=1)[0, class_idx].item()

# Grad-CAM on MobileNetV2 — last conv block gives the cleanest heatmaps
cam_model = mobilenet
target_layer = cam_model.features[-1]
gradcam = GradCAM(cam_model, target_layer)

def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Pick 8 random test images
idxs = np.random.RandomState(42).choice(len(test_224), size=8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.ravel(), idxs):
    x, y = test_224[i]
    xb = x.unsqueeze(0).to(device)
    cam, pred_idx, conf = gradcam(xb)
    img = denorm(x)
    ax.imshow(img)
    ax.imshow(cam, cmap='jet', alpha=0.45)
    true_lbl = 'Organic' if y == 0 else 'Recyclable'
    pred_lbl = 'Organic' if pred_idx == 0 else 'Recyclable'
    color = 'green' if pred_idx == y else 'red'
    ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', color=color, fontsize=10)
    ax.axis('off')
plt.suptitle('Grad-CAM heatmaps — MobileNetV2', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '10_gradcam.png'), dpi=150)
plt.show()

## Step 11 — Training curves (MobileNetV2 + ResNet18)

In [ ]:
def merge_hist(h1, h2):
    return {k: h1[k] + h2[k] for k in h1}

mob_hist = merge_hist(h1_mob, h2_mob)
res_hist = merge_hist(h1_res, h2_res)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, h in [('MobileNetV2', mob_hist), ('ResNet18', res_hist)]:
    axes[0].plot(h['train_loss'], label=f'{name} train')
    axes[0].plot(h['val_loss'],   label=f'{name} val', linestyle='--')
    axes[1].plot(h['train_acc'],  label=f'{name} train')
    axes[1].plot(h['val_acc'],    label=f'{name} val', linestyle='--')
axes[0].set_title('Loss');      axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Accuracy');  axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '11_training_curves_transfer.png'), dpi=150)
plt.show()

## Step 12 — Zip everything and download

In [ ]:
zip_path = '/content/pbl_continuation_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUT_DIR):
        zf.write(os.path.join(OUT_DIR, fn), arcname=fn)
print('Zipped to:', zip_path)
print('Contents:')
for fn in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, fn)
    print(f'  {fn:40s} {os.path.getsize(p)/1024:8.1f} KB')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('(not in Colab — skip auto-download):', e)